In [83]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from typing import Optional
import numpy as np
from tqdm import trange
from src.models_paper import MG_Daily, HMG_Intraday
from src.data import Yahoo_Downloader

In [95]:
N, F, n_heads = 10, 5, 4
daily = MG_Daily(N, F, n_heads, 1, d_model=4**4, n_classes=2, sigma_list=[10, 20, 30, 50])
data= Yahoo_Downloader(['^NDX'])

/Users/alexandre/ml_for_finance/src/squelette_models.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.scale = torch.tensor(1 / torch.sqrt(torch.tensor(d_model)))
[*********************100%***********************]  1 of 1 completed


In [96]:
tot = 0
for p in daily.parameters():
    tot+=p.numel()
print(f"total paramètres : {tot}")

total paramètres : 2199804


In [97]:
Xtrain, ytrain, Xval, yval, Xtest, ytest = data.compute_train_val_test(N, tensor=True)


In [98]:
daily(Xtrain[0:10])

tensor([[0.5571],
        [0.5408],
        [0.5439],
        [0.5376],
        [0.5345],
        [0.5363],
        [0.5565],
        [0.5623],
        [0.5565],
        [0.5500]], grad_fn=<SigmoidBackward0>)

In [99]:
from torch.utils.data import TensorDataset, DataLoader

gamma = 5
batch_size = 2**2
num_epochs = 50
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(daily.parameters(), lr=1e-4)

train_dataset = TensorDataset(Xtrain.float(), ytrain.float())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

for epoch in trange(num_epochs):
    k=0
    loss_ep=0

    for x_batch, y_batch in train_loader:
        k+=1
        B = x_batch.shape[0]
        outputs = daily(x_batch) 
        loss_penalty = daily.block1.mha._get_penalty() + daily.block2.mha._get_penalty() + daily.block3.mha._get_penalty()
        loss_val = loss_fn(outputs, y_batch) + gamma * loss_penalty
        loss_ep += loss_val.item()
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()

    with torch.no_grad():
        val_preds = daily(Xval)
        val_targets = yval
        val_loss = loss_fn(val_preds, val_targets)
        val_binary_preds = (val_preds > 0.5).float()  
        val_accuracy = (val_binary_preds == val_targets).float().mean()

    print(f"Epoch {epoch}, train_loss = {loss_ep/k:.4f} | val_loss = {val_loss.item():.4f} | val_acc = {val_accuracy.item():.4f}")


  2%|▏         | 1/50 [00:31<26:05, 31.96s/it]

Epoch 0, train_loss = 0.7749 | val_loss = 0.7441 | val_acc = 0.5689


  4%|▍         | 2/50 [01:02<24:54, 31.14s/it]

Epoch 1, train_loss = 0.7251 | val_loss = 0.6798 | val_acc = 0.5689


  6%|▌         | 3/50 [01:35<24:52, 31.75s/it]

...

Epoch 47, train_loss = 0.5564 | val_loss = 0.8235 | val_acc = 0.4756


 98%|█████████▊| 49/50 [26:59<00:30, 30.16s/it]

Epoch 48, train_loss = 0.5430 | val_loss = 0.7667 | val_acc = 0.5333


100%|██████████| 50/50 [27:32<00:00, 33.04s/it]

Epoch 49, train_loss = 0.5511 | val_loss = 0.7400 | val_acc = 0.5200


In [101]:
daily.block3.mha._get_penalty()

tensor(0.0009, grad_fn=<LinalgVectorNormBackward0>)

In [ ]:
outputs.shape

torch.Size([5, 5])

In [111]:
((daily(Xtrain)[:, :] > 0.5).float() == ytrain[:, :]).float().mean()


tensor(0.7127)

In [110]:
((daily(Xtest)[:, :] > 0.5).float() == ytest[:, :]).float().mean()


tensor(0.4756)

In [112]:
((daily(Xval)[:, :] > 0.5).float() == yval[:, :]).float().mean()


tensor(0.5200)

In [64]:
ytrain[5:11]

tensor([[1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.]])